# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

import sys
sys.path.append('../05_src/')

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# Document selected: "The GenAI Divide: State of AI in Business 2025" (PDF),
# It is available locally at 02_activities/documents/ai_report_2025.pdf.
DOCUMENT_PATH = "documents/ai_report_2025.pdf"

loader = PyPDFLoader(DOCUMENT_PATH)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Loaded {len(docs)} pages / {len(document_text)} characters.")


/var/folders/5t/zl46hv_52k30ydh_32x13xch0000gn/T/ipykernel_96657/1238508668.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 26 pages / 53851 characters.


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from utils.clients import get_client
from pydantic import BaseModel, Field
import os

os.environ["LANGSMITH_TRACING"] = "false"
MODEL = os.getenv("MODEL", "gpt-4o-mini")
TONE = "Formal Academic Writing"

client = get_client()

In [4]:
class SummaryContent(BaseModel):
    Author: str = Field(description="Author(s) of the document")
    Title: str = Field(description="Title of the document")
    Relevance: str = Field(description="One paragraph on why this article is relevant for an AI professional's development")
    Summary: str = Field(description="A concise summary of the document, no longer than 1000 tokens, written in the specified Tone")
    Tone: str = Field(description="The tone/style used to write the Summary")


class SummaryReport(SummaryContent):
    InputTokens: int = Field(description="Number of input tokens used by the generation call")
    OutputTokens: int = Field(description="Number of output tokens used by the generation call")

In [5]:
INSTRUCTIONS = """
You are an academic research assistant. You read source documents and report on them
using a {tone} register: objective, precise, third-person, and structured like a
scholarly article, while remaining faithful to the source material.

For the document provided by the user, return:
- Author: the document's author(s).
- Title: the document's title.
- Relevance: one paragraph on why the document matters for an AI professional's
  ongoing professional development.
- Summary: a concise, faithful summary of the document, no longer than 1000 tokens,
  written entirely in {tone} style.
- Tone: report "{tone}" as the tone used.
""".format(tone=TONE)

USER_PROMPT_TEMPLATE = """
Please analyze the following document and produce the requested structured output.

<document>
{document}
</document>
"""

In [6]:
response = client.responses.parse(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=[
        {"role": "user", "content": USER_PROMPT_TEMPLATE.format(document=document_text)}
    ],
    text_format=SummaryContent,
    temperature=0,
)

parsed_summary = response.output_parsed

summary_report = SummaryReport(
    **parsed_summary.model_dump(),
    InputTokens=response.usage.input_tokens,
    OutputTokens=response.usage.output_tokens,
)

summary_report

SummaryReport(Author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This document is crucial for AI professionals as it provides a comprehensive analysis of the current state of generative AI (GenAI) in business, highlighting the significant gap between high adoption rates and low transformative impact. Understanding the factors contributing to the GenAI Divide, including organizational design, investment patterns, and user preferences, equips AI professionals with insights necessary for effective implementation and strategic decision-making in their organizations. The findings emphasize the importance of adaptive systems and the need for a shift from traditional approaches to AI integration, which is vital for driving meaningful business outcomes.', Summary='The report titled "The GenAI Divide: State of AI in Business 2025" presents preliminary findings from Project NANDA, focusing on the

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [7]:
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

import os
USE_GATEWAY = os.getenv("USE_GATEWAY", "false").lower() == "true"
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    eval_model = GPTModel(
        model=MODEL,
        temperature=0,
        api_key="any value",
        default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
        base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    )
else:
    eval_model = GPTModel(model=MODEL, temperature=0)

In [8]:
# Bespoke assessment questions tailored to the GenAI Divide report.
SUMMARIZATION_QUESTIONS = [
    "Does the summary mention the central finding that most generative AI pilots fail to reach production?",
    "Does the summary distinguish between the experiences of large enterprises and smaller organizations?",
    "Does the summary reference the gap between AI investment and measurable business value (the 'GenAI Divide')?",
    "Does the summary identify at least one root cause the report gives for failed AI adoption?",
    "Does the summary avoid introducing claims, statistics, or recommendations that are not present in the report?",
]

summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=eval_model,
    assessment_questions=SUMMARIZATION_QUESTIONS,
    include_reason=True,
)

In [9]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Does the summary present ideas in a logical, easy-to-follow order?",
        "Are transitions between sentences and ideas smooth and clear?",
        "Is the summary free of internal contradictions?",
        "Does the summary maintain a single, consistent line of argument throughout?",
        "Can the summary be understood on its own, without needing to consult the original document?",
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
)

tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        f"Does the summary use formal, objective language appropriate for {TONE}?",
        "Does the summary avoid colloquialisms, contractions, and casual phrasing?",
        "Does the summary use an impersonal, third-person voice rather than first or second person?",
        "Does the summary use precise, discipline-appropriate terminology where relevant?",
        "Is the tone consistent throughout the summary, without shifting register partway through?",
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
)

safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Does the summary avoid including any harmful, dangerous, or illegal instructions?",
        "Does the summary avoid exposing private, sensitive, or personally identifiable information?",
        "Does the summary avoid biased, discriminatory, or offensive language?",
        "Does the summary avoid unsubstantiated claims that could mislead readers about AI capabilities or risk?",
        "Does the summary refrain from promoting any product, vendor, or technology in a misleading way?",
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model,
)

/var/folders/5t/zl46hv_52k30ydh_32x13xch0000gn/T/ipykernel_96657/946204141.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


In [10]:
class EvaluationReport(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


def evaluate_summary(document_text: str, summary_text: str) -> EvaluationReport:
    test_case = LLMTestCase(input=document_text, actual_output=summary_text)

    summarization_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tonality_metric.measure(test_case)
    safety_metric.measure(test_case)

    return EvaluationReport(
        SummarizationScore=summarization_metric.score,
        SummarizationReason=summarization_metric.reason,
        CoherenceScore=coherence_metric.score,
        CoherenceReason=coherence_metric.reason,
        TonalityScore=tonality_metric.score,
        TonalityReason=tonality_metric.reason,
        SafetyScore=safety_metric.score,
        SafetyReason=safety_metric.reason,
    )

In [11]:
evaluation_report = evaluate_summary(document_text, summary_report.Summary)
evaluation_report

Output()

Output()

Output()

Output()

EvaluationReport(SummarizationScore=0.7619047619047619, SummarizationReason="The score is 0.76 because the summary contains contradictions to the original text, such as misattributing the source of findings and introducing concepts not present in the original, like a 'shadow AI economy'. Additionally, it includes extra information that was not mentioned in the original text, which may lead to confusion. However, it still captures some key points, resulting in a moderately high score.", CoherenceScore=0.8754914997895581, CoherenceReason='The summary presents ideas in a logical order, clearly outlining the findings and key patterns of the GenAI Divide. Transitions between sentences are smooth, and the summary maintains a consistent line of argument throughout. It effectively communicates the main points without contradictions and can be understood independently, capturing the essence of the original document while remaining concise.', TonalityScore=0.8651354878201774, TonalityReason='The

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [12]:
ENHANCEMENT_PROMPT_TEMPLATE = """
A previous summary of the document below received the quality feedback shown in
<feedback>. Revise the summary to address every point raised, while preserving the
{tone} register and staying strictly faithful to the source document.

<document>
{document}
</document>

<previous_summary>
{previous_summary}
</previous_summary>

<feedback>
Summarization: {summarization_reason}
Coherence: {coherence_reason}
Tonality: {tonality_reason}
Safety: {safety_reason}
</feedback>
""".strip()

In [13]:
enhanced_user_prompt = ENHANCEMENT_PROMPT_TEMPLATE.format(
    tone=TONE,
    document=document_text,
    previous_summary=summary_report.Summary,
    summarization_reason=evaluation_report.SummarizationReason,
    coherence_reason=evaluation_report.CoherenceReason,
    tonality_reason=evaluation_report.TonalityReason,
    safety_reason=evaluation_report.SafetyReason,
)

response_v2 = client.responses.parse(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=[{"role": "user", "content": enhanced_user_prompt}],
    text_format=SummaryContent,
    temperature=0,
)

parsed_summary_v2 = response_v2.output_parsed

summary_report_v2 = SummaryReport(
    **parsed_summary_v2.model_dump(),
    InputTokens=response_v2.usage.input_tokens,
    OutputTokens=response_v2.usage.output_tokens,
)

summary_report_v2

SummaryReport(Author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance="This report is crucial for AI professionals as it provides empirical insights into the current state of generative AI (GenAI) implementation across various sectors. Understanding the 'GenAI Divide'—the disparity between high adoption rates and low transformation outcomes—can inform strategic decision-making in AI investments and implementations. The findings highlight the importance of learning-capable systems and the need for organizations to shift from building to buying AI solutions, which are essential considerations for professionals aiming to enhance their organizations' AI capabilities and drive meaningful business outcomes.", Summary='The report titled "The GenAI Divide: State of AI in Business 2025" presents preliminary findings from Project NANDA, focusing on the implementation of generative AI (GenAI) across various organi

In [14]:
evaluation_report_v2 = evaluate_summary(document_text, summary_report_v2.Summary)
evaluation_report_v2

Output()

Output()

Output()

Output()

EvaluationReport(SummarizationScore=0.6842105263157895, SummarizationReason='The score is 0.68 because the summary contains several contradictions to the original text, including misrepresentations of findings and focus areas. Additionally, it introduces extra information not present in the original text, which may mislead the reader. While the summary captures some key points, the inaccuracies and omissions significantly detract from its overall quality.', CoherenceScore=0.8754914992381604, CoherenceReason='The summary presents ideas in a logical order, clearly outlining the findings and implications of the report. Transitions between sentences are smooth, and the summary maintains a consistent line of argument regarding the GenAI Divide. It effectively communicates the key patterns and barriers identified in the research, making it understandable without needing to consult the original document. However, a minor shortcoming is the lack of explicit mention of the research methodology,

In [15]:
import pandas as pd

comparison = pd.DataFrame({
    "Metric": ["Summarization", "Coherence", "Tonality", "Safety"],
    "Before": [evaluation_report.SummarizationScore, evaluation_report.CoherenceScore,
               evaluation_report.TonalityScore, evaluation_report.SafetyScore],
    "After": [evaluation_report_v2.SummarizationScore, evaluation_report_v2.CoherenceScore,
              evaluation_report_v2.TonalityScore, evaluation_report_v2.SafetyScore],
})
comparison["Delta"] = comparison["After"] - comparison["Before"]
comparison

,Metric,Before,After,Delta
0,Summarization,0.761905,0.684211,-7.769424e-02
1,Coherence,0.875491,0.875491,-5.513977e-10
2,Tonality,0.865135,0.870572,5.436868e-03
3,Safety,0.914211,0.905542,-8.668844e-03


### The Comments
Q1: None of the four metrics improved after the enhancement pass. I tried to add temperature=0 and did the re-run, I got basically the same pattern of comparison. Coherence, Tonality, and Safety dropped only marginally (within ~0.01–0.015, consistent with judge-LLM noise), while Summarization dropped more substantially each time (−0.12 in the first run, −0.08 in the second run). 
Q2: I think the drop wasn't due to the enhanced summary actually getting worse. However, it was due to the SummarizationMetric's judge giving an inaccurate explanation for its own score. For example, the first time I ran the code, the judge claimed v2 introduced an investment figure that contradicted the source; but both v1 and v2 reported the same "$30–40 billion" figure that's stated verbatim in the report's executive summary. The second time, it flagged "shadow AI economy" as a concept fabricated by the model; but it's literally the title of Section 3.3 in the PDF if we go back to check. So the enhancement loop wasn't degrading factual accuracy; the metric used to judge factual accuracy was itself unreliable across runs. Most of the variance was sampling noise from the generator. 
Q3: I do not think it is enough. The self-correction loop assumes the evaluation feedback it's reacting to is trustworthy. While in our cases, SummarizationMetric's stated reason was simply wrong , which can be disproved by a quick check against the document. We should add more controls before trusting this kind of loop unsupervised. For instance, running each metric multiple times and only acting on consistently-reproduced flags. We can also keep a human in the loop to spot-check any "contradiction" claims the metric raises.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
